### ANN & Naive Bayes Classification — Car Price Range

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    ConfusionMatrixDisplay,
    roc_curve, auc                          # ★ NEW — for ROC/AUC analysis
)
from sklearn.inspection import permutation_importance  # ★ NEW — for ANN feature importance

### 1. Load Data

Loading the CSV file into a pandas Dataframe and printing the shape, data types, and basic stats. Given the rather large file, there's a complicity with relation to using Databricks and files of this size, in which we'll clean the file by data mining. 

In [0]:
df = pd.read_csv("car_prices.csv")
print(df.shape)
print(df.dtypes)
print(df.describe())


### 2. Drop rows with key missing values


We first remove any row with a missing value in any column - previously, I only checked four columns, which didn't deliver the best results (accuracy increased). Secondly, we removed any outliers with the IQR method (Interquartile Range) - defined as Q1 - 1.5 x IQR and Q3 + 1.5 x IQR - betweeen the spread of selling price and odomoeter, cutting anything below the 1.5x multiplier range. Through thtis, a car with extremely high miles with low value is filtred out for accuracy. Also, any car that's below $100 is removed. Cases such as this are considered as either typing or clerical error. 




In [0]:
def remove_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[column] >= lower_bound) & (data[column] <= upper_bound)]
 
df_cleaned = df.dropna().copy()
df_cleaned = remove_outliers(df_cleaned, 'sellingprice')
df_cleaned = remove_outliers(df_cleaned, 'odometer')
df_cleaned = df_cleaned[df_cleaned['sellingprice'] > 100]  # ★ NEW — remove junk/near-zero prices
 
print(f"Rows after cleaning: {len(df_cleaned)}")

### 3. Engineer Target Variable

What category, rather than raw number, can we predict using our models and the dataset? Splitting the target variables into categories such as Budget (under $10k), Mid ($10k–$30k), and Luxury ($30k+), allows for classification of relationships between economy vehicles, the standard used market, and premium/low-mileage cars - a reflection of the real used car market.  

In [0]:
def assign_price_range(price):
    if price < 10000:
        return "Budget"
    elif price < 30000:
        return "Mid"
    else:
        return "Luxury"
 
df_cleaned['price_range'] = df_cleaned['sellingprice'].apply(assign_price_range)
print(df_cleaned['price_range'].value_counts())

### 4. Feature Engineering

Determining the car age by subtracting it from the most recent year on the dataset. This is a reflection of the true depreciation of the vehicle, not the raw "year" that the vehcile was produced. Furthermore, the mileage by year is determined by dividiing the odomoter mileage by the car age we previously determined. This provides greater clarity on the real usage of the vehicle and creates an immediate distinction of how a vehcile features against its competitors.

In [0]:
df_cleaned['car_age'] = df_cleaned['year'].max() - df_cleaned['year']
df_cleaned['mileage_per_year'] = df_cleaned['odometer'] / (df_cleaned['car_age'] + 1)

### 5. Encode Categorical Features

Converting the categories into integers for reduced noise and improved machine learning - which doesn't work kindly with strings like "Ford" or "SUV." Additionally, the top 10 most common values are kept, and the others are filtered out as "other," to retain the most important categorical factors and further reduce the noise of the general population of the dataset. 

In [0]:
le = LabelEncoder()

In [0]:
categorical_cols = ['make', 'body', 'color', 'transmission']
for col in categorical_cols:
    top_10 = df_cleaned[col].value_counts().index[:10]
    df_cleaned[col] = df_cleaned[col].where(df_cleaned[col].isin(top_10), 'Other')
    df_cleaned[col + '_enc'] = le.fit_transform(df_cleaned[col].astype(str))

### 6. Define Features and Target

Formally chose predictors for the price range as the y axis. On the first attempt, the results were not optimal, so we went further by using features such as car age and mileage per year. By doing so, we revelaed that with r=0.98, the correlation between mmr and selling price are nearly perfect in relation to one another (this clearly violates the Naive Bayesian Classifier independence assumption).

In [0]:
features = [
    'year', 'car_age', 'condition', 'odometer', 'mileage_per_year',  # ★ car_age + mileage_per_year added
    'mmr', 'make_enc', 'body_enc', 'transmission_enc', 'color_enc'
]
 
X = df_cleaned[features]
y = df_cleaned['price_range']

### 7. Train/Test Split

Train and Test the data set by splitting it 80/20 - 80% for training the models, and 20% back for testing. The stratify=y argument ensured that each split contains the same proportion of Budget, Mid, and Luxury cars as the full dataset. Without stratification, one might accidentally get a test set with very few Budget cars, making evaluation less reliable.

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

###  8. Scale Features for ANN

Standard Scaler transforms every feature to have a mean of 0 and standard deviation of 1. This is only applied for the ANN — neural networks are sensitive to feature scale because large-valued features like odometer (in the hundreds of thousands) would otherwise dominate small-valued ones like condition (1–49). Naive Bayes is unaffected by scale, so the raw unscaled data is used for NBC.

In [0]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

### 9. Naive Bayes


Gaussian NB is trained on the unscaled data. It works by computing, for each class, the mean and variance of every feature — then when predicting, it uses Bayes' theorem to ask "given these feature values, which class is most probable?" The var_smoothing=1e-9 parameter adds a tiny amount to all variances to prevent division-by-zero errors for features with very low variance. Training is essentially instantaneous even on 400k rows.

In [0]:
gnb = GaussianNB(var_smoothing=1e-9)
gnb.fit(X_train, y_train)
y_pred_nb = gnb.predict(X_test)
 
print("=== Naive Bayes Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
print(classification_report(y_test, y_pred_nb))

### 10. ANN (Artificial Neural Network); (MLP Classifier)

The neural network has two hidden layers (128 then 64 neurons), uses ReLU activation to handle non-linearities, and is optimized with Adam — an adaptive learning rate algorithm well-suited to large datasets. alpha=0.001 is an L2 regularization penalty that discourages the model from over-fitting by keeping weights small. Unlike Naive Bayes, training here takes noticeable time as the model runs up to 300 passes through the training data adjusting its weights.

In [0]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=300,
    random_state=42
)
mlp.fit(X_train_scaled, y_train)
y_pred_mlp = mlp.predict(X_test_scaled)

print("\n=== ANN (MLP) Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_mlp):.4f}")
print(classification_report(y_test, y_pred_mlp))

Runs about 9 minutes total

### 11. Confusion Matrices

The Confusion Matrices grids show where each model makes its mistakes. 
Rows are the actual class, columns are what the model predicted.
A perfect model would have all values on the diagonal.
Off-diagonal values tell you the specific error pattern - like when a luxury vehcile is misclassified as a luxury range class. Regardless, this visual only relfects informative accuracy rather than numercial accuracy by showing the difficulty in differentations.  

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_nb),
    display_labels=gnb.classes_).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Naive Bayes — Confusion Matrix")

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_mlp),
    display_labels=mlp.classes_).plot(ax=axes[1], colorbar=False)
axes[1].set_title("ANN (MLP) — Confusion Matrix")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

### 12. Accuracy Comparison Bar Chart

Simple visual to summarize the accuracy numbers in a side-by-side comparison. ~89 percent versus ~95 percent Naive Bayes versus ANN, respectively. 

In [0]:
models = ['Naive Bayes', 'ANN (MLP)']
accuracies = [accuracy_score(y_test, y_pred_nb),
              accuracy_score(y_test, y_pred_mlp)]

plt.figure(figsize=(7, 4))
bars = plt.bar(models, accuracies, color=['#4C72B0', '#DD8452'], width=0.4)
plt.ylim(0, 1.05)
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison")
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{acc:.4f}", ha='center', fontsize=11)
plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=150)
plt.show()

### 13. Class Distribution

How man y 

In [0]:
plt.figure(figsize=(6, 4))
df['price_range'].value_counts().plot(kind='bar', color=['#4C72B0','#DD8452','#55A868'])
plt.title("Price Range Class Distribution")
plt.xlabel("Price Range")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()

### 14. Correlation Heatmap

In [0]:
plt.figure(figsize=(10, 8))
corr_cols = ['sellingprice', 'odometer', 'mmr', 'condition', 'year']
correlation_matrix = df_cleaned[corr_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.show()

### 15. Depreciation Curve

In [0]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_cleaned, x='car_age', y='sellingprice')
plt.title("Depreciation Curve: Average Price by Car Age")
plt.xlabel("Car Age (years)")
plt.ylabel("Average Selling Price ($)")
plt.grid(True)
plt.tight_layout()
plt.savefig("depreciation_curve.png", dpi=150)
plt.show()

### 16. ROC/AUC Analysis


In [0]:
median_price = df_cleaned['sellingprice'].median()
y_binary = (df_cleaned['sellingprice'] > median_price).astype(int)
 
X_b = df_cleaned[features]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y_binary, test_size=0.2, random_state=42
)
X_train_b_scaled = scaler.fit_transform(X_train_b)
X_test_b_scaled  = scaler.transform(X_test_b)
 
mlp_binary = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation='relu',
    solver='adam', alpha=0.001, max_iter=300, random_state=42
)
mlp_binary.fit(X_train_b_scaled, y_train_b)
 
gnb_binary = GaussianNB(var_smoothing=1e-9)
gnb_binary.fit(X_train_b, y_train_b)
 
plt.figure(figsize=(10, 6))
 
for model, x_data, label, color in [
    (mlp_binary, X_test_b_scaled, 'ANN (MLP)',    'purple'),
    (gnb_binary, X_test_b,        'Naive Bayes',   'steelblue'),
]:
    probs = model.predict_proba(x_data)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_b, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{label} (AUC = {roc_auc:.3f})', color=color)
 
plt.plot([0, 1], [0, 1], 'k--', label='Random Baseline (AUC = 0.500)')
plt.title('ROC Curve Comparison — ANN vs Naive Bayes')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150)
plt.show()

### 17. Feature Importance

In [0]:
from sklearn.tree import DecisionTreeClassifier
 
dtree = DecisionTreeClassifier(max_depth=5, random_state=42)
dtree.fit(X_train_b, y_train_b)
 
dt_importance = pd.DataFrame({
    'Feature': features,
    'Importance': dtree.feature_importances_
}).sort_values('Importance', ascending=False)
 
nn_perm = permutation_importance(
    mlp_binary, X_test_b_scaled, y_test_b, n_repeats=5, random_state=42
)
nn_importance = pd.DataFrame({
    'Feature': features,
    'Importance': nn_perm.importances_mean
}).sort_values('Importance', ascending=False)
 
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
 
sns.barplot(x='Importance', y='Feature', data=dt_importance,
            ax=ax[0], hue='Feature', palette='viridis', legend=False)
ax[0].set_title('Decision Tree — Gini Feature Importance')
 
sns.barplot(x='Importance', y='Feature', data=nn_importance,
            ax=ax[1], hue='Feature', palette='plasma', legend=False)
ax[1].set_title('ANN — Permutation Feature Importance')
 
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()
 
print("\nDecision Tree — Top 5 Features:")
print(dt_importance.head(5).to_string(index=False))
 
print("\nANN (Permutation) — Top 5 Features:")
print(nn_importance.head(5).to_string(index=False))